# Withdrawals

In [ ]:
#| default_exp withdraw

In [ ]:
#| export

from dataclasses import dataclass
from typing import Optional
from sugar.pool import LiquidityPool
from sugar.position import Position

In [ ]:
#| export

@dataclass(frozen=True)
class Withdrawal:
    """Input to `chain.withdraw`. Built from a `Position` via `Withdrawal.from_position`;
    Sugar's positions reader already provides current-price amounts so no on-chain quote
    step is needed."""
    pool: LiquidityPool
    # LP tokens to burn (basic) / liquidity to decrease (CL)
    liquidity: int
    # expected amounts out; slippage min computed at execution
    amount_token0: int
    amount_token1: int
    # CL only — the NFT tokenId
    position_id: Optional[int] = None
    # CL only — when True, append nfpm.burn(position_id) to the multicall to clean up the empty NFT
    burn: bool = False

    def __post_init__(self):
        if self.pool.is_cl and self.position_id is None:
            raise ValueError("CL Withdrawal requires position_id")
        if not self.pool.is_cl and self.position_id is not None:
            raise ValueError("basic Withdrawal must not set position_id")
        if self.liquidity <= 0: raise ValueError("liquidity must be positive")
        if self.burn and not self.pool.is_cl: raise ValueError("burn is CL-only")

    @classmethod
    def from_position(cls, position: Position, *, fraction: float = 1.0, burn: bool = False) -> "Withdrawal":
        """Linear scaling: `liquidity_out = position.liquidity * fraction`. Raw token amounts
        come from Sugar's snapshot (`position.amount_token0/1`); slippage on `chain.withdraw`
        protects against price drift between snapshot and execution.

        `burn=True` requires `fraction=1.0` — partial closes can't burn since liquidity remains
        on the NFT. `fraction=1.0` short-circuits the float multiplication for exact integer
        liquidity (important for very large V3 L values)."""
        if not 0 < fraction <= 1: raise ValueError("fraction must be in (0, 1]")
        if burn and fraction != 1.0: raise ValueError("burn requires fraction=1.0 (full close)")
        if position.liquidity == 0: raise ValueError("position has no liquidity to withdraw")
        if fraction == 1.0:
            liquidity, a0, a1 = position.liquidity, position.amount_token0, position.amount_token1
        else:
            liquidity = int(position.liquidity * fraction)
            if liquidity == 0: raise ValueError("fraction too small: computed liquidity is 0")
            a0 = int(position.amount_token0 * fraction)
            a1 = int(position.amount_token1 * fraction)
        return cls(
            pool=position.pool, liquidity=liquidity,
            amount_token0=a0, amount_token1=a1,
            position_id=position.id if position.is_cl else None,
            burn=burn,
        )

In [ ]:
# Construction + guards. The on-chain withdraw happy path is not covered by automated tests,
# matching the existing pattern for `swap` / `deposit`. Run manually on supersim.
from types import SimpleNamespace
from fastcore.test import test_eq, test_fail

basic_pool = SimpleNamespace(is_cl=False)
cl_pool = SimpleNamespace(is_cl=True)

# Withdrawal post-init guards
w = Withdrawal(pool=basic_pool, liquidity=100, amount_token0=10, amount_token1=20)
test_eq((w.position_id, w.burn), (None, False))
w = Withdrawal(pool=cl_pool, liquidity=100, amount_token0=10, amount_token1=20, position_id=42)
test_eq(w.position_id, 42)
test_fail(lambda: Withdrawal(pool=cl_pool, liquidity=100, amount_token0=10, amount_token1=20))
test_fail(lambda: Withdrawal(pool=basic_pool, liquidity=100, amount_token0=10, amount_token1=20, position_id=1))
test_fail(lambda: Withdrawal(pool=basic_pool, liquidity=0, amount_token0=10, amount_token1=20))
test_fail(lambda: Withdrawal(pool=basic_pool, liquidity=-1, amount_token0=10, amount_token1=20))
# burn is CL-only
test_fail(lambda: Withdrawal(pool=basic_pool, liquidity=100, amount_token0=10, amount_token1=20, burn=True))

# from_position with raw int Position fields
basic_pos = SimpleNamespace(is_cl=False, liquidity=100, pool=basic_pool, id=0,
                            amount_token0=1_000_000, amount_token1=2_000_000)
cl_pos = SimpleNamespace(is_cl=True, liquidity=1000, pool=cl_pool, id=42,
                         amount_token0=1_000_000, amount_token1=2_000_000)

w = Withdrawal.from_position(basic_pos)
test_eq((w.liquidity, w.amount_token0, w.amount_token1, w.position_id, w.burn), (100, 1_000_000, 2_000_000, None, False))

w = Withdrawal.from_position(cl_pos)
test_eq((w.liquidity, w.amount_token0, w.amount_token1, w.position_id, w.burn), (1000, 1_000_000, 2_000_000, 42, False))

# fraction=1.0 short-circuits — exact identity for huge L (no float precision loss)
huge_pos = SimpleNamespace(is_cl=True, liquidity=10**30, pool=cl_pool, id=1,
                           amount_token0=10**30, amount_token1=2*10**30)
w = Withdrawal.from_position(huge_pos)
test_eq(w.liquidity, 10**30)
test_eq(w.amount_token0, 10**30)
test_eq(w.amount_token1, 2*10**30)

# fraction=0.5 halves
w = Withdrawal.from_position(cl_pos, fraction=0.5)
test_eq((w.liquidity, w.amount_token0, w.amount_token1), (500, 500_000, 1_000_000))

# burn=True requires fraction=1.0
w = Withdrawal.from_position(cl_pos, burn=True)
test_eq(w.burn, True)
test_fail(lambda: Withdrawal.from_position(cl_pos, fraction=0.5, burn=True))
test_fail(lambda: Withdrawal.from_position(cl_pos, fraction=0.999, burn=True))
# burn on basic pool rejected
test_fail(lambda: Withdrawal.from_position(basic_pos, burn=True))

# fraction outside (0, 1] rejected
test_fail(lambda: Withdrawal.from_position(cl_pos, fraction=0))
test_fail(lambda: Withdrawal.from_position(cl_pos, fraction=1.1))
test_fail(lambda: Withdrawal.from_position(cl_pos, fraction=-0.1))

# empty position rejected
empty_pos = SimpleNamespace(is_cl=True, liquidity=0, pool=cl_pool, id=1, amount_token0=10, amount_token1=20)
test_fail(lambda: Withdrawal.from_position(empty_pos))

# partial withdrawal that rounds liquidity to zero rejected
tiny = SimpleNamespace(is_cl=True, liquidity=10, pool=cl_pool, id=1, amount_token0=10, amount_token1=20)
test_fail(lambda: Withdrawal.from_position(tiny, fraction=0.001))

# one-sided CL (out of range) passes through with zero on the empty leg
one_sided = SimpleNamespace(is_cl=True, liquidity=1000, pool=cl_pool, id=1, amount_token0=0, amount_token1=2_000_000)
w = Withdrawal.from_position(one_sided)
test_eq((w.amount_token0, w.amount_token1), (0, 2_000_000))

In [ ]:
#| hide

import nbdev; nbdev.nbdev_export()